# Benchmark: ImageFolder (rclone mount)

This notebook benchmarks a PyTorch `DataLoader` reading Food11 using `torchvision.datasets.ImageFolder`.

In this run, the dataset path inside the container (default `/mnt/Food-11`) is backed by an rclone-mounted object store on the VM host.


In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print('torch:', torch.__version__)


## Configuration


In [ ]:
DATA_ROOT = os.environ.get('FOOD11_DATA_DIR', '/mnt/Food-11')
SPLIT = os.environ.get('FOOD11_SPLIT', 'evaluation')

BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '64'))
NUM_WORKERS_LIST = [0, 2, 4, 8]

WARMUP_BATCHES = 10
MEASURE_BATCHES = 50
MAX_SAMPLES = int(os.environ.get('MAX_SAMPLES', '0'))  # 0 means no cap

print('DATA_ROOT:', DATA_ROOT)
print('SPLIT:', SPLIT)
print('BATCH_SIZE:', BATCH_SIZE)
print('NUM_WORKERS_LIST:', NUM_WORKERS_LIST)
print('WARMUP_BATCHES:', WARMUP_BATCHES)
print('MEASURE_BATCHES:', MEASURE_BATCHES)
print('MAX_SAMPLES:', MAX_SAMPLES)


## Dataset


In [ ]:
split_dir = os.path.join(DATA_ROOT, SPLIT)
if not os.path.isdir(split_dir):
    raise FileNotFoundError(f'Missing split directory: {split_dir} (is the dataset mounted?)')

# Minimal transform: decode + convert to tensor.
transform = transforms.Compose([transforms.ToTensor()])

dataset = datasets.ImageFolder(root=split_dir, transform=transform)

if MAX_SAMPLES and MAX_SAMPLES < len(dataset):
    dataset = torch.utils.data.Subset(dataset, list(range(MAX_SAMPLES)))

print('num_samples:', len(dataset))


## Run benchmark


In [ ]:
results = []

for num_workers in NUM_WORKERS_LIST:
    loader_kwargs = {
        'batch_size': BATCH_SIZE,
        'shuffle': False,
        'num_workers': num_workers,
        'pin_memory': False,
        'drop_last': False,
    }
    if num_workers > 0:
        loader_kwargs['prefetch_factor'] = 2
        loader_kwargs['persistent_workers'] = True

    loader = DataLoader(dataset, **loader_kwargs)
    it = iter(loader)

    t0 = time.perf_counter()
    x0, y0 = next(it)
    t1 = time.perf_counter()
    time_to_first_batch_s = t1 - t0

    for _ in range(WARMUP_BATCHES):
        try:
            _ = next(it)
        except StopIteration:
            break

    num_batches = 0
    num_items = 0
    t_start = time.perf_counter()
    for _ in range(MEASURE_BATCHES):
        try:
            x, y = next(it)
        except StopIteration:
            break
        num_batches += 1
        num_items += int(y.shape[0])
    t_end = time.perf_counter()

    wall_s = t_end - t_start
    imgs_per_s = (num_items / wall_s) if wall_s > 0 else float('nan')
    batches_per_s = (num_batches / wall_s) if wall_s > 0 else float('nan')
    avg_batch_s = (wall_s / num_batches) if num_batches > 0 else None

    results.append({
        'num_workers': num_workers,
        'batch_size': BATCH_SIZE,
        'time_to_first_batch_s': time_to_first_batch_s,
        'measured_batches': num_batches,
        'measured_items': num_items,
        'wall_s': wall_s,
        'imgs_per_s': imgs_per_s,
        'batches_per_s': batches_per_s,
        'avg_batch_s': avg_batch_s,
    })

results


## Print results


In [ ]:
print('split:', SPLIT)
print('batch_size:', BATCH_SIZE)
print('warmup_batches:', WARMUP_BATCHES, 'measure_batches:', MEASURE_BATCHES)
print()

for r in results:
    avg_batch_s = r['avg_batch_s']
    avg_batch_s_str = 'nan' if avg_batch_s is None else f"{avg_batch_s:.4f}"
    print(
        'workers=', r['num_workers'],
        'imgs/s=', f"{r['imgs_per_s']:.2f}",
        'batches/s=', f"{r['batches_per_s']:.2f}",
        'first_batch_s=', f"{r['time_to_first_batch_s']:.3f}",
        'avg_batch_s=', avg_batch_s_str,
    )

imgs_per_s_values = [r['imgs_per_s'] for r in results if r['measured_batches'] > 0]
if imgs_per_s_values:
    best = max(results, key=lambda x: x['imgs_per_s'])
    mean_imgs_per_s = sum(imgs_per_s_values) / len(imgs_per_s_values)
    print()
    print('aggregate_mean_imgs_per_s:', round(mean_imgs_per_s, 2))
    print('aggregate_best_imgs_per_s:', round(best['imgs_per_s'], 2), 'at num_workers=', best['num_workers'])
else:
    print('No measured batches; check dataset and settings.')


## Save results


In [ ]:
out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_path = out_dir / f'imagefolder_rclone_mount_{stamp}.json'
payload = {
    'benchmark': 'imagefolder_rclone_mount',
    'timestamp_utc': stamp,
    'data_root': DATA_ROOT,
    'split': SPLIT,
    'batch_size': BATCH_SIZE,
    'warmup_batches': WARMUP_BATCHES,
    'measure_batches': MEASURE_BATCHES,
    'max_samples': MAX_SAMPLES,
    'results': results,
}
out_path.write_text(json.dumps(payload, indent=2))
print('Wrote:', out_path)
